In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import math

In [ ]:
gdf_boundaries = gpd.read_file(r"Population\Ward_Boundary_gj.geojson")
buildings_gdf = gpd.read_parquet(r"output_merged_datasets_Churu.parquet")

In [ ]:
buildings_gdf

In [ ]:
gdf_boundaries

In [ ]:
churu_res_buildings = buildings_gdf[buildings_gdf["prediction"] == "Residential"]
churu_other_buildings = buildings_gdf[buildings_gdf["prediction"] != "Residential"]
churu_res_buildings["geometry2"] = buildings_gdf["geometry"]

In [ ]:
churu_res_buildings = gpd.GeoDataFrame(
    churu_res_buildings,
    geometry=gpd.points_from_xy(churu_res_buildings["longitude"], churu_res_buildings["latitude"]),
    crs="EPSG:4326"
)
churu_res_buildings

In [ ]:
churu_res_buildings = churu_res_buildings.to_crs(gdf_boundaries.crs)
churu_res_buildings

In [ ]:
joined = gpd.sjoin(churu_res_buildings, gdf_boundaries, how="left", predicate="within")
joined_wo_nan = joined[joined['Name'].notna()]
churu_res_buildings = joined_wo_nan

In [ ]:
churu_res_buildings.columns

In [ ]:
#file from MHT
population_churu = pd.read_excel(r"Population\Churu Population (1).xlsx")
population_churu

In [ ]:
#total ground floor area
total_gfa = churu_res_buildings['gfa_in_meters'].sum()
total_gfa

In [ ]:
#average gfa per inhabitant 
total_population = 171979.010
average_area_per_inhabitant = total_gfa / total_population
average_area_per_inhabitant

In [ ]:
#per gfam - churu total
churu_res_buildings["inhabitants_whole_churu"] = (churu_res_buildings["gfa_in_meters"] / total_gfa) * total_population
churu_res_buildings['Name'].astype(int)
churu_res_buildings['Name'].astype(str)

In [ ]:
# #not rounded per person
# churu_res_buildings = churu_res_buildings.copy()
# churu_res_buildings["inhabitants_with_ward"] = np.nan


# for i in range(1, 61):
#     df_i = churu_res_buildings[churu_res_buildings["Name"] == str(i)]  # if Name is string
#     # df_i = churu_res_buildings[churu_res_buildings["Name"] == i]     # if Name is numeric
    
#     pop_df_i = population_churu[population_churu["Ward Number"] == i]
    
#     if pop_df_i.empty or df_i.empty:
#         continue
    
#     total_pop_i = pop_df_i["Pop_2025"].values[0]
#     total_gfa_i = df_i["gfa_in_meters"].sum()
    
#     inhabitants = (df_i["gfa_in_meters"] / total_gfa_i) * total_pop_i
    
#     churu_res_buildings.loc[df_i.index, "inhabitants_with_ward"] = inhabitants
#churu_res_buildings

In [ ]:
#without decimal numbers
churu_res_buildings = churu_res_buildings.copy()
churu_res_buildings["inhabitants_with_integer_estimate"] = np.nan

for ward in range(1, 61):
    try:
        mask = churu_res_buildings["Name"].astype(int) == ward
    except:
        mask = churu_res_buildings["Name"].astype(str) == str(ward)
    
    df_ward = churu_res_buildings.loc[mask].copy()
    
    try:
        pop_row = population_churu[population_churu["Ward Number"].astype(int) == ward]
    except:
        pop_row = population_churu[population_churu["Ward Number"].astype(str) == str(ward)]
    
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    if total_gfa == 0:
        continue

    df_ward["ROcc"] = (df_ward["gfa_in_meters"] / total_gfa) * total_pop
    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]

    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        churu_res_buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1.0)

    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1

    churu_res_buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values


In [ ]:
churu_res_buildings['inhabitants_with_integer_estimate']


In [ ]:
#without decimal numbers + informal settlements constant
churu_res_buildings = churu_res_buildings.copy()
churu_res_buildings["inhabitants_with_integer_informal"] = np.nan

informal_multiplier = 4

for ward in range(1, 61):
    mask = churu_res_buildings["Name"].astype(str) == str(ward)
    df_ward = churu_res_buildings.loc[mask].copy()
    pop_row = population_churu[population_churu["Ward Number"] == ward]
    
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    if total_gfa == 0:
        continue
    
    df_ward["weight"] = np.where(df_ward["settlement_clasification"].str.lower() == "informal", informal_multiplier, 1)
    df_ward["weighted_gfa"] = df_ward["gfa_in_meters"] * df_ward["weight"]
    total_weighted_gfa = df_ward["weighted_gfa"].sum()
    

    df_ward["ROcc"] = (df_ward["weighted_gfa"] / total_weighted_gfa) * total_pop
    

    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]
    
    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        churu_res_buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1)
    
    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1
    

    churu_res_buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values


In [ ]:
#adding back geometry + nonres buildings
churu_res_buildings['geometry'] = churu_res_buildings['geometry2'] 
merged = pd.concat([churu_res_buildings, churu_other_buildings], ignore_index=True)

In [ ]:
merged_gdf = gpd.GeoDataFrame(merged, geometry='geometry')
merged_gdf = merged_gdf.to_crs(epsg=4326)
merged_gdf.to_parquet("churu_with_population2.parquet")